# Module 2 — Environment Setup

# Project Sahur: The Base Notebook

Welcome to the official notebook for **Project Sahur**.

This notebook is your continuously integrated workspace. We are starting with the environment setup (Module 2). In future modules, we will load the model (Module 3), explore its internals (Module 4), engineer our dataset (Module 5), and fine-tune Sahur's personality into the weights (Module 6).

**Runtime Requirement:** Before running anything, go to `Runtime → Change runtime type → Hardware accelerator → T4 GPU`.

## 1. Installing Dependencies (Lessons 2.1, 2.5–2.10)

We install the full Unsloth stack, which automatically pulls in the Hugging Face ecosystem:
- **`torch`** — The engine. Every neural network operation runs through PyTorch. (Lesson 2.3)
- **`transformers`** — Load any model from Hugging Face with `AutoModel`. (Lesson 2.5)
- **`datasets`** — Load JSONL/CSV/Parquet training data into Apache Arrow. (Lesson 2.4)
- **`peft`** — LoRA and QLoRA adapter injection. (Lesson 2.6)
- **`trl`** — High-level SFT and RLHF trainers. (Lesson 2.7)
- **`accelerate`** — Automatic device placement (`device_map="auto"`). (Lesson 2.8)
- **`bitsandbytes`** — 4-bit NF4 quantization to compress models. (Lesson 2.9)
- **`unsloth`** — Fused CUDA kernels for 2–5x faster fine-tuning. (Lesson 2.10)

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27"
!pip install "trl<0.9.0" peft accelerate bitsandbytes datasets sentencepiece safetensors

## 2. Verifying CUDA & GPU (Lesson 2.2)

CUDA is NVIDIA's software platform that allows PyTorch to communicate with the GPU. Without it, all tensor math runs on the CPU (hundreds of times slower). Let's confirm PyTorch can see the Colab T4.

In [ ]:
import torch

print("--- CUDA & GPU Verification ---")
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version:     {torch.version.cuda}")
    print(f"GPU Device:       {torch.cuda.get_device_name(0)}")
    vram_total = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"Total VRAM:       {vram_total:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## 3. Verifying the Hugging Face Ecosystem (Lessons 2.4–2.10)

Let's confirm all our critical libraries imported correctly and print their versions.

In [ ]:
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("--- Library Versions ---")
print(f"transformers:   {transformers.__version__}")
print(f"datasets:       {datasets.__version__}")
print(f"peft:           {peft.__version__}")
print(f"trl:            {trl.__version__}")
print(f"accelerate:     {accelerate.__version__}")
print(f"bitsandbytes:   {bitsandbytes.__version__}")
print("\n✅ All libraries loaded successfully!")

## 4. PyTorch Tensor Basics (Lesson 2.3)

Tensors are the fundamental data structure of deep learning. Every weight, every activation, and every gradient inside Qwen3 is a tensor. Let's verify that we can create tensors and move them between CPU and GPU.

In [ ]:
# Create a tensor on CPU
cpu_tensor = torch.randn(3, 4)  # 3 rows, 4 columns of random floats
print(f"CPU Tensor Shape: {cpu_tensor.shape}")
print(f"CPU Tensor Device: {cpu_tensor.device}")

# Move it to GPU
gpu_tensor = cpu_tensor.to("cuda")
print(f"GPU Tensor Device: {gpu_tensor.device}")

# Matrix multiplication on GPU (this is what every Transformer layer does)
weight_matrix = torch.randn(4, 2).to("cuda")
result = gpu_tensor @ weight_matrix  # Y = X · W
print(f"Result Shape: {result.shape} (3 tokens × 2 dimensions)")
print("\n✅ GPU tensor math working perfectly!")

## 5. Connecting to the Hugging Face Hub (Lesson 2.4)

Hugging Face is where every major open-source model lives. We log in so we can later push our fine-tuned Sahur model back to the hub.

In [ ]:
from huggingface_hub import notebook_login

# This will prompt a widget where you can paste your Hugging Face Access Token.
# Get your token at: https://huggingface.co/settings/tokens
notebook_login()

---

✅ **Module 2 Complete.** Our laboratory is fully operational. We have:
- Installed the complete Unsloth + Hugging Face stack
- Verified CUDA, GPU detection, and VRAM availability
- Confirmed all library versions
- Demonstrated GPU tensor math
- Connected to the Hugging Face Hub

In **Module 3**, we will download Qwen3-0.6B, load its 600M parameters into VRAM, build the tokenizer pipeline, and run our first inference.

---
# Module 3 — Loading the Model

We are now going to take Qwen3-0.6B out of its box, load its 600 million parameters into VRAM, build the tokenizer pipeline, explore sampling strategies, and ultimately build a professional streaming inference engine.

## 6. Loading the Model Weights (Lesson 3.1)

The `.safetensors` file sitting on Hugging Face's servers contains 600 million floating-point numbers. We need to:
1. Download the file.
2. Build an empty PyTorch skeleton from `config.json`.
3. Fill the skeleton with the actual weights ("hydration").
4. Transfer everything into the T4 GPU's VRAM.

We use `AutoModelForCausalLM` (not `AutoModel`) because we need the **Language Modeling Head** — the final layer that outputs logits for next-token prediction. We load in FP16 to halve the VRAM footprint from ~2.4 GB to ~1.2 GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings("ignore")

MODEL_ID = "Qwen/Qwen3-0.6B"

print(f"📥 Downloading and loading {MODEL_ID}...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,   # Half precision: 2 bytes per param instead of 4
    device_map="auto"            # accelerate places weights on the GPU automatically
)

vram_allocated = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"✅ Model loaded!")
print(f"GPU VRAM Allocated: {vram_allocated:.2f} GB")

### Inspecting the Architecture

Let's look at the actual PyTorch object. You will see the `embed_tokens` embedding layer, 28 `Qwen2DecoderLayer` blocks (each containing `q_proj`, `k_proj`, `v_proj` attention projections and SwiGLU FFN), and the `lm_head` at the bottom.

In [ ]:
# Print the full architecture tree
print(model)

## 7. Loading the Tokenizer (Lesson 3.2)

The model only understands integer tensor IDs. The tokenizer translates human text into token IDs using Byte-Pair Encoding (BPE). Qwen3 has a vocabulary of **151,646 unique tokens**.

The tokenizer also handles **ChatML formatting** — wrapping messages with `<|im_start|>` and `<|im_end|>` special tokens that teach the model who is speaking.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Fix padding if not set (required for batch generation)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded!")
print(f"Vocabulary Size:    {tokenizer.vocab_size}")
print(f"EOS Token:          {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"Pad Token:          {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

### Exploring Tokenization

Let's see how the tokenizer splits text into subword tokens. Notice how common words stay intact, while rarer words get split into subword fragments.

In [ ]:
test_text = "Yo Sahur, explain attention mechanisms!"

# Encode: text → token IDs
token_ids = tokenizer.encode(test_text)
print(f"Text:      '{test_text}'")
print(f"Token IDs: {token_ids}")
print(f"Num Tokens: {len(token_ids)}")

# Show each token individually
print("\nToken Breakdown:")
for tid in token_ids:
    print(f"  ID {tid:>6} → {repr(tokenizer.decode([tid]))}")

# Decode: token IDs → text
decoded = tokenizer.decode(token_ids)
print(f"\nDecoded:   '{decoded}'")

### ChatML Formatting with `apply_chat_template`

In real inference, we don't manually tokenize raw strings. We structure the conversation into roles (`system`, `user`, `assistant`) and let the tokenizer wrap them in ChatML. The `add_generation_prompt=True` flag appends `<|im_start|>assistant\n` to signal the model that it should now generate a response.

In [ ]:
messages = [
    {"role": "system", "content": "You are Sahur, a Gen-Z coding assistant."},
    {"role": "user", "content": "how do I reverse a list in python?"}
]

# Convert to ChatML string (not tokenized yet)
chatml_string = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("--- Raw ChatML String ---")
print(chatml_string)

## 8. First Generation — The Autoregressive Loop (Lesson 3.3)

An LLM generates text **one token at a time**. `model.generate()` automates this loop:
1. Forward pass through 28 Transformer layers → outputs 151,646 logits.
2. Select the winning token ID.
3. Append to the sequence.
4. Check stop conditions (`<|im_end|>` or `max_new_tokens`).
5. Loop back.

In [ ]:
# Tokenize the ChatML prompt and move to GPU
inputs = tokenizer(chatml_string, return_tensors="pt").to(model.device)
prompt_length = inputs["input_ids"].shape[1]

print(f"Prompt token count: {prompt_length}")
print(f"input_ids shape:    {inputs['input_ids'].shape}")
print(f"attention_mask shape: {inputs['attention_mask'].shape}")

# Run the autoregressive loop (greedy decoding, no sampling)
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False   # Greedy: always pick the highest probability token
    )

# Slice off the prompt tokens to get ONLY the generated response
response_ids = output_ids[0][prompt_length:]
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)

print(f"\nGenerated {len(response_ids)} new tokens.")
print(f"\n🤖 Sahur: {response_text}")

## 9. Sampling Strategies (Lessons 3.4–3.7)

Greedy decoding (always picking the highest-probability token) produces deterministic but often boring, repetitive text. Real inference uses **sampling** — randomly selecting from the probability distribution.

- **Temperature** (3.5): Controls randomness. Low (0.2) = focused/deterministic. High (1.5) = creative/chaotic.
- **Top-K** (3.6): Only sample from the K highest-probability tokens. Cuts off the long tail.
- **Top-P / Nucleus** (3.7): Only sample from tokens whose cumulative probability reaches P. Adapts dynamically.

In [ ]:
def generate_with_params(prompt_text, temperature=0.8, top_k=50, top_p=0.9, max_tokens=100):
    """Generate a response with specific sampling parameters."""
    msgs = [
        {"role": "system", "content": "You are Sahur, a Gen-Z coding assistant."},
        {"role": "user", "content": prompt_text}
    ]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(formatted, return_tensors="pt").to(model.device)
    plen = inp["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(out[0][plen:], skip_special_tokens=True)

### Experiment: Temperature Comparison

Run the same prompt at different temperatures to see how randomness affects the output.

In [ ]:
prompt = "write a one-liner joke about Python programming"

print("--- Temperature 0.2 (Very Focused) ---")
print(generate_with_params(prompt, temperature=0.2))

print("\n--- Temperature 0.8 (Balanced) ---")
print(generate_with_params(prompt, temperature=0.8))

print("\n--- Temperature 1.5 (Very Creative) ---")
print(generate_with_params(prompt, temperature=1.5))

## 10. Streaming Inference Engine (Lessons 3.8–3.9)

Standard `model.generate()` is a blocking function — you wait seconds for the full output. `TextStreamer` intercepts the autoregressive loop and prints each token the instant it's generated, just like ChatGPT.

We also enable `use_cache=True` for the **KV Cache** (Lesson 3.8), which prevents the model from recomputing attention for all previous tokens on every step.

In [ ]:
from transformers import TextStreamer

class SahurInferenceEngine:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

        # skip_prompt=True: Don't echo the user's question back
        self.streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        # Golden Sampling Config (from Lessons 3.4-3.7)
        self.gen_config = {
            "max_new_tokens": 200,
            "do_sample": True,
            "temperature": 0.8,
            "top_k": 50,
            "top_p": 0.90,
            "repetition_penalty": 1.15,
            "pad_token_id": tokenizer.pad_token_id,
            "eos_token_id": tokenizer.eos_token_id,
            "use_cache": True  # KV Cache ON (Lesson 3.8)
        }
        print("✅ SahurInferenceEngine ready.")

    def chat(self, user_message):
        messages = [
            {"role": "system", "content": "You are Sahur, a hyper-responsive Gen Z AI assistant."},
            {"role": "user", "content": user_message}
        ]
        prompt = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        print("\n🤖 Sahur: ", end="")
        with torch.no_grad():
            self.model.generate(**inputs, streamer=self.streamer, **self.gen_config)
        print("\n" + "─"*50)

engine = SahurInferenceEngine(model, tokenizer)

### Interactive Chat Loop

This cell starts a continuous chat session. Type your messages and watch the tokens stream in real-time. Type `exit` to end.

In [ ]:
print("💬 Chat session started! (Type 'exit' to end)")
print("─"*50)

while True:
    try:
        user_input = input("🧑 You: ")
        if user_input.lower() in ['exit', 'quit']:
            print("👋 Peace out!")
            break
        if not user_input.strip():
            continue
        engine.chat(user_input)
    except KeyboardInterrupt:
        print("\n👋 Session interrupted.")
        break